# **Reading Data**

In [ ]:
!pip install dash

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dash import Dash, html, dcc
import plotly.express as px
import datetime
from sklearn import model_selection
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import acf,pacf
from statsmodels.tsa.seasonal import seasonal_decompose
import itertools
import statsmodels.api as sm
from sklearn.preprocessing import MinMaxScaler

In [ ]:
train = pd.read_csv(r'dataset/train.csv', parse_dates=['Date'],index_col='Date', low_memory=False)
store = pd.read_csv(r'dataset/store.csv')

# **Data Cleaning and EDA**

***Train Dataset***

In [ ]:
train.shape

In [ ]:
train.info()

In [ ]:
train.head()



In [ ]:
train.isnull().sum()

In [ ]:
store.info()

In [ ]:
# Extract Year, Month, Day columns
train['Year'] = train.index.year
train['Month'] = train.index.month
train['Day'] = train.index.day
train['WeekofYear'] = train.index.isocalendar().week
train.head()

In [ ]:
train.index[:2]

In [ ]:
# Create a new feature 'SalesPerCustomer' to measure average sales per customer
train['SalesPerCustomer'] = train['Sales']/train['Customers']
train.sample(10)

In [ ]:
# Fill all missing values in the dataset with 0
train.fillna(value=0,inplace=True)

In [ ]:
#Check for outlier
pd.set_option('display.float_format', lambda x: '%.2f' % x)
train['Sales'].describe()

In [ ]:
train.info()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=train, x='Sales')
plt.title('Boxplot of Sales (Outlier Detection)')
plt.xlabel('Sales')
plt.show()


In [ ]:
# Detect outliers in the 'Sales' column using the Interquartile Range (IQR) method
# Q1: 25th percentile, Q3: 75th percentile
# IQR = Q3 - Q1
# Any value below (Q1 - 1.5*IQR) or above (Q3 + 1.5*IQR) is considered an outlier

Q1 = train['Sales'].quantile(0.25)
Q3 = train['Sales'].quantile(0.75)
IQR = Q3 - Q1

# Calculate lower and upper bounds for detecting outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter out the rows that have outlier values in Sales
outliers = train[(train['Sales'] < lower_bound) | (train['Sales'] > upper_bound)]

# Display the number and percentage of outliers
print(f"Number of outliers: {len(outliers)}")
print(f"Percentage of outliers: {len(outliers)/len(train)*100:.2f}%")


***Store Dataset***

In [ ]:
store.shape

In [ ]:
store.info()

In [ ]:
store.head(10)

In [ ]:
store.isnull().sum()

## **Data Merging**

In [ ]:
# Step 1: Reset index so Date becomes a regular column before merging
train_reset = train.reset_index()   # Date moves from index → column

# Step 2: Merge — Date is now a regular column and survives the merge
store_merged = pd.merge(train_reset, store, on='Store', how='left')

# Step 3: Restore Date as the index after merging
store_merged = store_merged.set_index('Date')
store_merged.index = pd.to_datetime(store_merged.index)
store_merged = store_merged.sort_index()

In [ ]:
store_merged.info()

In [ ]:
store_merged['WeekofYear'].nunique()

In [ ]:
store_merged['StoreType'].unique()

In [ ]:
store_merged['StateHoliday'].unique()

In [ ]:
store_merged['StoreType'].unique()

In [ ]:
store_merged['Assortment'].unique()

In [ ]:
store_merged['PromoInterval'].unique()

In [ ]:
#asking data is their any null values
store_merged.isnull().sum()

### **Handling missing data through imputation**

In [ ]:
store_merged['PromoInterval'].fillna('0',inplace=True)

In [ ]:
store_merged['PromoInterval'] = store_merged['PromoInterval'].astype(str)

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
          'Jul', 'Aug', 'Sept', 'Oct', 'Nov', 'Dec']

for month in months:
    store_merged[f'Promo_{month}'] = store_merged['PromoInterval'].apply(lambda x: 1 if month in x else 0)

In [ ]:
store_merged.drop(columns=['PromoInterval'], inplace=True)

In [ ]:
store_merged.sample(10)

In [ ]:
# Handle CompetitionDistance
store_merged['CompetitionDistance'] = store_merged['CompetitionDistance'].fillna(200000)

# Logical zeros for stores with no competition or promo
store_merged.loc[store_merged['CompetitionDistance'] == 200000,
           ['CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear']] = 0

store_merged.loc[store_merged['Promo2'] == 0,
           ['Promo2SinceWeek', 'Promo2SinceYear']] = 0

# Iterative imputation for remaining valid gaps
cols_to_impute = ['CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear',
                  'Promo2SinceWeek', 'Promo2SinceYear']

imputer = IterativeImputer(random_state=42)
store_merged[cols_to_impute] = imputer.fit_transform(store_merged[cols_to_impute])


In [ ]:
store_merged['CompetitionOpenSinceYear'].sample(10)

In [ ]:
store_merged['CompetitionOpenSinceMonth'].sample(10)

In [ ]:
store_merged['CompetitionOpenSinceMonth'] = store_merged['CompetitionOpenSinceMonth'].astype(int)
store_merged['CompetitionOpenSinceYear'] = store_merged['CompetitionOpenSinceYear'].astype(int)

In [ ]:
store_merged['CompetitionOpenSinceMonth'].sample(10)

In [ ]:
store_merged['CompetitionOpenSinceMonth'].sample(10)

In [ ]:
# Clean StateHoliday column
# Replace the string '0' with numeric 0 to standardize data types
store_merged['StateHoliday'].replace(to_replace='0',value=0,inplace=True)
store_merged["StateHoliday"].unique()

In [ ]:
# StateHoliday , StoreType , Assortment are still category columns and need to transform with one hot encoding/label encoder for training.
store_merged.info()

## **EDA**

#### **How many days sales equal zero.**

In [ ]:
#check how many sales value has zero
print(store_merged[store_merged['Sales'] == 0].shape[0])

In [ ]:
store_merged.sample(10)

In [ ]:
#investigate the reason why we have many sales equal zero
zero_sales = store_merged[store_merged['Sales'] == 0]
group_days = zero_sales.groupby('DayOfWeek')['Sales'].size().reset_index(name='ZeroSalesCount')
group_days

In [ ]:
# Group zero sales by DayOfWeek
zero_sales = store_merged[store_merged['Sales'] == 0]
group_days = zero_sales.groupby('DayOfWeek')['Sales'].size().reset_index(name='ZeroSalesCount')

# Sort values by day to make the chart more readable
group_days = group_days.sort_values(by='DayOfWeek')

# Plot bar chart
plt.figure(figsize=(10,6))
sns.barplot(x='DayOfWeek', y='ZeroSalesCount', data=group_days, palette='viridis')

plt.title('Number of Zero Sales per Day of the Week', fontsize=14)
plt.xlabel('Day of Week', fontsize=12)
plt.ylabel('Zero Sales Count', fontsize=12)
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


We found that the maximum day with highest zero sales was the 7th day

### **General plots and Visualizations**
- Sales distribution by year
- Monthly trend sales by year.
- Total sales by month.
- Average sales per day of the week.
- Daily sales trend
- Effect of Promotions on Average Sales.
- Total Sales by Store Type.
- The impact of the state holidays on average sales.
- The number of open and closed days for each store type
- Correlation Heatmap
- Number of stores by year.
- Number of stores opened each month.

In [ ]:
#Sales distribution by year
fig = px.pie(store_merged, values='Sales', names='Year',title="Sales by year")
fig.show()

Stores are open more often in 2013 and 2014 than 2015 which is even more explained in the sales being highest at 2013 and lowest at 2015.

In [ ]:
# Monthly trend sales by year
monthly_sales = store_merged.groupby(['Year','Month'])['Sales'].sum().reset_index()
plt.figure(figsize=(14,5))
sns.lineplot(data=monthly_sales, x='Month', y='Sales', hue='Year', marker='o')
plt.title(' Monthly Sales Trend by Year')
plt.xlabel('Month')
plt.ylabel('Total Sales')
plt.show()


In [ ]:
# Total sales by month to identify seasonal patterns

plt.figure(figsize=(6,5))
store_merged.groupby('Month')['Sales'].sum().plot(kind='bar', color='skyblue')
plt.title('Total Sales by Month', fontsize=14)
plt.ylabel('Total Sales')
plt.xlabel('Month')
plt.xticks(rotation=0)
plt.show()


In [ ]:
fig = px.pie(store_merged, values='Sales', names='Month',title='sales per month')
fig.show()

Sales are highest in march and lowest in septemper and just like said previously sales decrease in the last 5 months beginning with august.

In [ ]:
# Average sales per day of the week
##“Which days of the week have higher average sales?”

plt.figure(figsize=(6,5))
store_merged.groupby('DayOfWeek')['Sales'].mean().plot(kind='bar', color='lightgreen')
plt.title('Average Sales by Day of the Week', fontsize=14)
plt.ylabel('Average Sales')
plt.xlabel('Day of the Week')
plt.xticks(rotation=0)
plt.show()


In [ ]:
# Daily sales trend
daily_sales = store_merged.groupby('Day')['Sales'].sum().reset_index()
plt.figure(figsize=(14,5))
sns.lineplot(data=daily_sales, x='Day', y='Sales')
plt.title(' Daily Sales Trend')
plt.xlabel('Day')
plt.ylabel('Total Sales')
plt.show()



## Compare sales with and without promo

In [ ]:
promo_sales = store_merged.groupby(['Promo', 'StoreType'])['Sales'].mean().reset_index()
print(promo_sales)

In [ ]:

plt.figure(figsize=(6,5))
sns.barplot(data=promo_sales, x='Promo', y='Sales', palette='viridis')
plt.title('Effect of Promotions on Average Sales')
plt.ylabel('Average Sales')
plt.xticks([0, 1], ['No Promo', 'Promo'])
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

# Use 'hue' to split bars by Promo status within each StoreType x-axis category
sns.barplot(data=store_merged, x='StoreType', y='Sales', hue='Promo', palette='viridis', order=['a', 'b', 'c', 'd'])

plt.title('Effect of Promotions on Sales by Store Type')
plt.ylabel('Average Sales')
plt.xlabel('Store Type')

# Optional: Customize legend to be clearer
plt.legend(title='Promotion Status', labels=['No Promo (0)', 'Promo Active (1)'])

plt.show()


In [ ]:
# Step 1: Prepare aggregated data
comp_promo = store_merged.groupby(['StoreType', 'Promo']).agg(
    AvgSales=('Sales', 'mean'),
    AvgCompetitionDistance=('CompetitionDistance', 'mean')
).reset_index()

store_order = ['a', 'b', 'c', 'd']

# Step 2: Create figure with two Y-axes
fig, ax1 = plt.subplots(figsize=(12, 6))

# --- Bar chart: Average Sales split by Promo (on ax1) ---
sns.barplot(
    data=comp_promo,
    x='StoreType', y='AvgSales',
    hue='Promo',
    palette='viridis',
    order=store_order,
    ax=ax1
)
ax1.set_ylabel('Average Sales', fontsize=12)
ax1.set_xlabel('Store Type', fontsize=12)
ax1.set_title('Effect of Promo & Competition Distance on Sales by Store Type', fontsize=14)
ax1.legend(title='Promotion Status', labels=['No Promo (0)', 'Promo Active (1)'])

# --- Line chart: Avg Competition Distance per StoreType (on ax2) ---
ax2 = ax1.twinx()

comp_dist = store_merged.groupby('StoreType')['CompetitionDistance'].mean().reindex(store_order)

ax2.plot(
    store_order,
    comp_dist.values,
    color='tomato',
    marker='o',
    linewidth=2.5,
    linestyle='--',
    label='Avg Competition Distance'
)
ax2.set_ylabel('Avg Competition Distance (meters)', fontsize=12, color='tomato')
ax2.tick_params(axis='y', labelcolor='tomato')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=store_merged,
    x='Customers',
    y='Sales',
    hue='StoreType',        # color by store type
    palette='viridis',
    alpha=0.6
)

plt.title('Customers vs Sales by Store Type')
plt.xlabel('Number of Customers')
plt.ylabel('Sales')
plt.legend(title='Store Type')
plt.tight_layout()
plt.show()


In [ ]:
total_sales = store_merged['Sales'].sum()
avg_daily_sales = store_merged.groupby(store_merged.index)['Sales'].sum().mean()
num_stores = store_merged['Store'].nunique()

fig, ax = plt.subplots(figsize=(8, 3))
ax.axis('off')  # hide axes

# Card 1
ax.text(
    0.05, 0.6,
    f"Total Sales\n{total_sales:,.0f}",
    fontsize=14, weight='bold',
    bbox=dict(boxstyle="round,pad=0.5", fc="#e3f2fd", ec="#90caf9")
)

# Card 2
ax.text(
    0.37, 0.6,
    f"Avg Daily Sales\n{avg_daily_sales:,.0f}",
    fontsize=14, weight='bold',
    bbox=dict(boxstyle="round,pad=0.5", fc="#e8f5e9", ec="#a5d6a7")
)

# Card 3
ax.text(
    0.69, 0.6,
    f"Active Stores\n{num_stores}",
    fontsize=14, weight='bold',
    bbox=dict(boxstyle="round,pad=0.5", fc="#fff3e0", ec="#ffcc80")
)

plt.tight_layout()
plt.show()


### Total Sales by Store Type
 **“Which store type contributes most to sales?”**  


- `a` → Large stores (hypermarkets or big supermarkets) with high sales.  
- `b` → Medium-sized supermarkets with moderate traffic.  
- `c` → Small neighborhood stores with lower sales but frequent customers.  
- `d` → Specialized or discount stores with varying performance.


In [ ]:
# Group the data by StoreType and sum Sales, Customers, and SalesPerCustomer
plt.figure(figsize=(6,5))
store_merged.groupby('StoreType')[['Sales', 'Customers', 'SalesPerCustomer']]\
    .sum().sort_values('Sales', ascending=False)\
    .plot(kind='bar', figsize=(10,6))
plt.title('Sales by Store Type', fontsize=14)
plt.ylabel('Total')
plt.xlabel('Store Type')
plt.xticks(rotation=0)
plt.show()



we see that store type a has the most number of sales and customers while store type b has the least

In [ ]:
#  bar plot for total sales by store type for a clearer view.
store_type_sales = store_merged.groupby('StoreType')['Sales'].sum().reset_index().sort_values('Sales', ascending=False)

plt.figure(figsize=(6,5))
sns.barplot(data=store_type_sales, x='StoreType', y='Sales', palette='Blues')
plt.title('Total Sales by Store Type')
plt.xlabel('Store Type')
plt.ylabel('Total Sales')
plt.show()

In [ ]:
#Which stores have the highest and lowest sales?
store_merged.groupby('StoreType')[['Customers', 'Sales', 'SalesPerCustomer']].sum().sort_values('Sales', ascending=False)



###  The impact of state holidays on average sales to understand seasonal

- `0` → No holiday (a regular business day).
- `a` → Public holiday (e.g., national or state-level holidays).
- `b`→ Easter holiday (a special holiday period around Easter).
- `c`→ Christmas holiday (the holiday period around Christmas).

In [ ]:
# Visualization of the impact of state holidays on average sales .

plt.figure(figsize=(8,5))
sns.barplot(data=store_merged, x='StateHoliday', y='Sales', estimator='mean', palette='viridis')
plt.title('Average Sales by State Holiday Type')
plt.xlabel('State Holiday Type')
plt.ylabel('Average Sales')
plt.show()

In [ ]:
#Showing number of the days where the store is opened and the sales = 0.
store_merged[(store_merged.Open == 1) & (store_merged.Sales == 0)].shape[0]

There is a small number of days in which the store was open and yet there were no sales.

**Count the number of open and closed days for each store type**

In [ ]:
open_days = store_merged.groupby(['StoreType', 'Open']).size().reset_index(name='Days')
open_days.head(10)

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(data=open_days, x='StoreType', y='Days', hue='Open')

plt.title('Days Open vs Closed by Store Type')
plt.xlabel('Store Type')
plt.ylabel('Number of Days')
plt.legend(title='Open', labels=['Closed (0)', 'Open (1)'])
plt.show()

we can see that store b had the least number of open days and is far less represented in the dataset compared to the other stores followed closely by store c

In [ ]:
plt.figure(figsize=(12, 8))

corr_matrix = store_merged[[
    'Sales', 'Customers', 'SalesPerCustomer',
    'Promo', 'DayOfWeek', 'Open',
    'CompetitionDistance'
]].corr()

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='coolwarm',
    fmt=".2f",
    linewidths=0.5,
    annot_kws={"size": 10}
)

plt.title('Correlation Heatmap (with Competition Distance)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
#Number of stores opened by year
fig = px.pie(store_merged, values='Open', names='Year')
fig.show()

stores have been opening less each year with the highest number of store opened in 2013 and lowest being in 2015

In [ ]:
#Number of stores opened each month.
plt.figure(figsize=(10, 6));
sns.countplot(x='Month', hue='Open', data=store_merged);
plt.title("stores open each month")
plt.show()

stores open on average consistently each month with the exception of the last 5 months where there is a notable drop off in general datapoints.

#  **Preprocessing data for models**


In [ ]:
store_merge = store_merged.copy()

In [ ]:
# Define numeric and categorical columns
numeric_cols = [
    'Sales', 'Customers', 'Day', 'Month',
    'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear',
    'Promo2SinceWeek', 'Promo2SinceYear', 'Year','WeekofYear','DayOfWeek'
]

categorical_cols = ['StoreType', 'Assortment', 'StateHoliday']

In [ ]:
store_merged = pd.get_dummies(store_merged, columns=categorical_cols, prefix=categorical_cols)

In [ ]:
store_merged[numeric_cols] = store_merged[numeric_cols].astype('float32')
store_merged = store_merged.astype({col: 'float32' for col in store_merged.select_dtypes('uint8').columns})

In [ ]:
drop_cols = ['Store', 'Sales', 'Customers', 'SalesPerCustomer']
X = store_merged.drop(columns=drop_cols)
y = store_merged['Sales']

In [ ]:
print(X.shape,y.shape)

In [ ]:
store_merged.info()

# **Time Series Model**

##  Testing data for stationarity


In [ ]:
sales_a = train[train.Store == 2]['Sales']
sales_b = train[train.Store == 85]['Sales'].sort_index(ascending = True)
sales_c = train[train.Store == 1]['Sales']
sales_d = train[train.Store == 13]['Sales']

frame, (ax1, ax2, ax3, ax4) = plt.subplots(4, figsize = (20, 16))

# Visualize Trend
sales_a.resample('w').sum().plot(ax = ax1)
sales_b.resample('w').sum().plot(ax = ax2)
sales_c.resample('w').sum().plot(ax = ax3)
sales_d.resample('w').sum().plot(ax = ax4)


In [ ]:
def test_stationarity(timeseries):
    # Determine rolling statestics
    roll_mean = timeseries.rolling(window=7).mean()
    roll_std = timeseries.rolling(window=7).std()

    # plotting rolling statestics
    plt.subplots(figsize = (16, 6))
    orginal = plt.plot(timeseries.resample('W').mean(), color='blue',linewidth= 3, label='Orginal')
    roll_mean = plt.plot(roll_mean.resample('W').mean(), color='red',linewidth= 3, label='Rolling Mean')
    roll_mean = plt.plot(roll_std.resample('W').mean(), color='green',linewidth= 3, label='Rolling Std')

    plt.legend(loc='best')
    plt.show()

    # Performing Dickey-Fuller test
    print('Results of Dickey-Fuller test:')
    result= adfuller(timeseries, autolag='AIC')

    print('ADF Statistics: %f' %result[0])
    print('P-value: %f' %result[1])
    print('Critical Values:')
    for key, value in result[4].items():
        print(key, value)

In [ ]:
test_stationarity(sales_a)

In [ ]:
test_stationarity(sales_b)

In [ ]:
test_stationarity(sales_c)

In [ ]:
test_stationarity(sales_d)

From above charts we could deduct that, mean and variance of the data do not change most of the time. So, we do not compute any transformation as the data is already stationary.


In [ ]:
def plot_timeseries(sales,StoreType):

    fig, axes = plt.subplots(2, 1, sharex=True, sharey=False)
    fig.set_figheight(6)
    fig.set_figwidth(20)

    decomposition= seasonal_decompose(sales, model = 'additive',period=365)

    estimated_trend = decomposition.trend
    estimated_seasonal = decomposition.seasonal
    estimated_residual = decomposition.resid

    axes[1].plot(estimated_seasonal, 'g', label='Seasonality')
    axes[1].legend(loc='upper left');

    axes[0].plot(estimated_trend, label='Trend')
    axes[0].legend(loc='upper left');

    plt.title('Decomposition Plots')

In [ ]:
plot_timeseries(sales_a, 'a')

In [ ]:
plot_timeseries(sales_b, 'b')

In [ ]:
plot_timeseries(sales_c, 'c')

In [ ]:
plot_timeseries(sales_d, 'd')

From the above plots, we can see that there is seasonality and trends present in our data. So, we'll use forecasting models that take both of these factors into consideration.

In [ ]:
def auto_corr(sales):
    lag_acf = acf(sales, nlags=30)
    lag_pacf = pacf(sales, nlags=20, method='ols')

    plt.figure(figsize=(10,5))

    plt.subplot(121)
    plt.plot(lag_acf)
    plt.axhline(y=0, linestyle='--', color='red')
    plt.axhline(y=1.96/np.sqrt(len(sales)), linestyle='--', color='red')
    plt.axhline(y=-1.96/np.sqrt(len(sales)), linestyle='--', color='red')
    plt.title('Autocorrelation (ACF)')

    plt.subplot(122)
    plt.plot(lag_pacf)
    plt.axhline(y=0, linestyle='--', color='red')
    plt.axhline(y=1.96/np.sqrt(len(sales)), linestyle='--', color='red')
    plt.axhline(y=-1.96/np.sqrt(len(sales)), linestyle='--', color='red')
    plt.title('Partial Autocorrelation (PACF)')

    plt.tight_layout()
    plt.show()


In [ ]:
auto_corr(sales_a)


In [ ]:
auto_corr(sales_b)

In [ ]:
auto_corr(sales_c)

In [ ]:
auto_corr(sales_d)

In [ ]:
numeric_train = train.select_dtypes(include='number')
train_arima = numeric_train.resample('W').mean()
train_arima = train_arima[['Sales']]
train_arima.plot()

### First model : SARIMAX

In [ ]:
import itertools
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# -----------------------------------------------
# 0. Log + MinMax scaling on weekly series
#    (apply on full train_arima, but fit on train segment later)
# -----------------------------------------------
# train_arima currently has a weekly resampled Sales column
# shape: index=Date (weekly), column 'Sales'
assert 'Sales' in train_arima.columns, "train_arima must contain 'Sales'"

# We'll compute the time boundaries first to know which part is train
total_days = (train_arima.index.max() - train_arima.index.min()).days
train_end  = train_arima.index.min() + pd.Timedelta(days=int(total_days * 0.60))
val_end    = train_arima.index.min() + pd.Timedelta(days=int(total_days * 0.80))

train_mask = train_arima.index <= train_end
val_mask   = (train_arima.index > train_end) & (train_arima.index <= val_end)
test_mask  = train_arima.index > val_end

# log1p transform
train_arima = train_arima.copy()
train_arima['Sales_log'] = np.log1p(train_arima['Sales'].astype('float32'))

# Fit MinMaxScaler on train part of Sales_log only
sarima_scaler = MinMaxScaler()
log_train = train_arima.loc[train_mask, 'Sales_log'].values.reshape(-1, 1)
sarima_scaler.fit(log_train)

# Apply scaler to all three parts
train_arima['Sales_scaled'] = sarima_scaler.transform(
    train_arima['Sales_log'].values.reshape(-1, 1)
).ravel()

# -----------------------------------------------
# 1. Time-Based Split (60% Train / 20% Val / 20% Test)
#    using the scaled series
# -----------------------------------------------
train_data = train_arima.loc[train_mask, ['Sales_scaled']]
val_data   = train_arima.loc[val_mask,   ['Sales_scaled']]
test_data  = train_arima.loc[test_mask,  ['Sales_scaled']]

train_val_data = train_arima.loc[train_mask | val_mask, ['Sales_scaled']]

print(f"Train  : {train_data.index.min().date()} → {train_data.index.max().date()} | {len(train_data)} weeks")
print(f"Val    : {val_data.index.min().date()}   → {val_data.index.max().date()}   | {len(val_data)} weeks")
print(f"Test   : {test_data.index.min().date()}  → {test_data.index.max().date()}  | {len(test_data)} weeks")

In [ ]:
import itertools
import numpy as np
import pandas as pd
from math import sqrt
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# -----------------------------------------------
# 1. Time-Based Split (60% Train / 20% Val / 20% Test)
#    using the scaled series (Sales_scaled)
# -----------------------------------------------
total_days = (train_arima.index.max() - train_arima.index.min()).days

train_end = train_arima.index.min() + pd.Timedelta(days=int(total_days * 0.60))
val_end   = train_arima.index.min() + pd.Timedelta(days=int(total_days * 0.80))

train_data     = train_arima.loc[train_arima.index <= train_end, ['Sales_scaled']]
val_data       = train_arima.loc[(train_arima.index > train_end) & (train_arima.index <= val_end),
                                 ['Sales_scaled']]
test_data      = train_arima.loc[train_arima.index > val_end, ['Sales_scaled']]
train_val_data = train_arima.loc[train_arima.index <= val_end, ['Sales_scaled']]

print(f"Train  : {train_data.index.min().date()} → {train_data.index.max().date()} | {len(train_data)} weeks")
print(f"Val    : {val_data.index.min().date()}   → {val_data.index.max().date()}   | {len(val_data)} weeks")
print(f"Test   : {test_data.index.min().date()}  → {test_data.index.max().date()}  | {len(test_data)} weeks")

# -----------------------------------------------
# 2. Define SARIMA Parameter Ranges
# -----------------------------------------------
p = d = q = range(0, 2)
P = range(0, 2)
D = range(0, 2)
Q = range(0, 2)
S_values = [7, 14, 21, 29, 52]

pdq          = list(itertools.product(p, d, q))
PDQ          = list(itertools.product(P, D, Q))
seasonal_pdq = [(x[0], x[1], x[2], s) for s in S_values for x in PDQ]

# -----------------------------------------------
# 3. Grid Search (Trained on train_data, Evaluated on val_data)
#    endog is univariate: train_data['Sales_scaled']
# -----------------------------------------------
results_grid = []

for param in pdq:
    for param_seasonal in seasonal_pdq:
        try:
            mod = sm.tsa.statespace.SARIMAX(
                train_data['Sales_scaled'],          # univariate endog
                order=param,
                seasonal_order=param_seasonal,
                enforce_stationarity=False,
                enforce_invertibility=False
            )
            mod_results = mod.fit(disp=False)

            if np.any(np.isinf(mod_results.bse)) or np.any(mod_results.bse < 0):
                continue

            pred_val = mod_results.forecast(steps=len(val_data))
            rmse_val = np.sqrt(
                mean_squared_error(val_data['Sales_scaled'], pred_val)
            )

            results_grid.append({
                'order':          param,
                'seasonal_order': param_seasonal,
                'S':              param_seasonal[3],
                'AIC':            round(mod_results.aic, 2),
                'RMSE_val':       round(rmse_val, 2)
            })

            print(f'SARIMAX{param}x{param_seasonal} | AIC: {mod_results.aic:.2f} | Val RMSE: {rmse_val:.2f}')

        except Exception:
            continue

# -----------------------------------------------
# 4. Select Best Model
# -----------------------------------------------
results_df = pd.DataFrame(results_grid)

best_aic  = results_df.loc[results_df['AIC'].idxmin()]
best_rmse = results_df.loc[results_df['RMSE_val'].idxmin()]

print('\n' + '='*55)
print('Best AIC Model:')
print(f"   order={best_aic['order']}  seasonal_order={best_aic['seasonal_order']}")
print(f"   AIC: {best_aic['AIC']}  |  Val RMSE: {best_aic['RMSE_val']}")

print('\nBest RMSE Model:')
print(f"   order={best_rmse['order']}  seasonal_order={best_rmse['seasonal_order']}")
print(f"   AIC: {best_rmse['AIC']}  |  Val RMSE: {best_rmse['RMSE_val']}")

print('\nTop 5 Models (sorted by Val RMSE):')
print(results_df.sort_values('RMSE_val').head().to_string(index=False))

# -----------------------------------------------
# 5. Retrain Best Model on Train+Val, Evaluate on Test
# -----------------------------------------------
best_order          = tuple(best_rmse['order'])
best_seasonal_order = tuple(best_rmse['seasonal_order'])

# OPTION A: Use best parameters found by grid search (recommended)
# OPTION B: Override with your own validated parameters — uncomment below
# best_order          = (0, 1, 1)
# best_seasonal_order = (1, 0, 0, 52)

final_model = sm.tsa.statespace.SARIMAX(
    train_val_data['Sales_scaled'],      # univariate endog
    order=best_order,
    seasonal_order=best_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)
results_sarima = final_model.fit(disp=False)
print(results_sarima.summary().tables[1])

# Forecast on test_data and compute final RMSE in scaled space
pred_test = results_sarima.forecast(steps=len(test_data))
rmse_test = np.sqrt(
    mean_squared_error(test_data['Sales_scaled'], pred_test)
)

print('\n' + '='*55)
print(f'Final Test RMSE (scaled, last 20% time period): {rmse_test:.2f}')

In [ ]:
# -----------------------------------------------
# Fit the Final SARIMA Model on Train + Val Data
# (train_val_data was defined in the previous step as the first 80% by time)
# -----------------------------------------------
final_model = sm.tsa.statespace.SARIMAX(
    train_val_data,           # Sales_scaled on train+val
    order=best_order,
    seasonal_order=best_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)
results_sarima = final_model.fit(disp=False)
print(results_sarima.summary().tables[1])

In [ ]:
# Checking diagnostic plots
results_sarima.plot_diagnostics(figsize=(16, 10))
plt.show()

In [ ]:
from math import sqrt
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

# -----------------------------------------------
# 1. In-Sample Prediction on Train+Val Data (scaled)
# -----------------------------------------------
# Start prediction from the beginning of val_data to see how well
# the model fits the period it was trained on
pred = results_sarima.get_prediction(
    start=val_data.index.min(),
    end=train_val_data.index.max(),
    dynamic=False
)

pred_ci = pred.conf_int()

# -----------------------------------------------
# 2. Plot Train, Val, and In-Sample Prediction (scaled)
# -----------------------------------------------
ax = train_val_data['Sales_scaled'].plot(
    label='Train + Val (Observed, scaled)', figsize=(15, 7)
)
pred.predicted_mean.plot(
    ax=ax, label='One Step Ahead Forecast (scaled)', alpha=1, color='orange'
)

ax.fill_between(
    pred_ci.index,
    pred_ci.iloc[:, 0],
    pred_ci.iloc[:, 1],
    color='r', alpha=0.1
)

ax.set_xlabel('Date')
ax.set_ylabel('Sales_scaled')
plt.legend()
plt.show()

# -----------------------------------------------
# 3. RMSE on Validation Period Only (scaled)
# -----------------------------------------------
val_forecasted = pred.predicted_mean.loc[val_data.index.min():val_data.index.max()]
val_truth      = val_data['Sales_scaled']

rms_val_scaled = sqrt(mean_squared_error(val_truth, val_forecasted))
print('RMSE on Validation Data (scaled) = ', rms_val_scaled)

# -----------------------------------------------
# 4. Out-of-Sample Forecast on Test Data (scaled)
# -----------------------------------------------
forecast_obj = results_sarima.get_forecast(steps=len(test_data))
pred_mean    = forecast_obj.predicted_mean
pred_ci_test = forecast_obj.conf_int()

# Align forecast index with test_data index
pred_mean.index    = test_data.index
pred_ci_test.index = test_data.index

# -----------------------------------------------
# 5. Plot Test Forecast vs Actual Test Data (scaled)
# -----------------------------------------------
ax2 = train_val_data['Sales_scaled']['2014':].plot(
    label='Train + Val (Observed, scaled)', figsize=(15, 7)
)
test_data['Sales_scaled'].plot(ax=ax2, label='Test (Observed, scaled)', color='green')
pred_mean.plot(ax=ax2, label='Forecast on Test Data (scaled)', color='orange')

ax2.fill_between(
    pred_ci_test.index,
    pred_ci_test.iloc[:, 0],
    pred_ci_test.iloc[:, 1],
    color='r', alpha=0.1
)

ax2.set_xlabel('Date')
ax2.set_ylabel('Sales_scaled')
plt.legend()
plt.show()

# -----------------------------------------------
# 6. Final RMSE on Test Data (scaled)
# -----------------------------------------------
rms_test_scaled = sqrt(
    mean_squared_error(test_data['Sales_scaled'], pred_mean)
)
print('RMSE on Test Data (scaled) = ', rms_test_scaled)

# **Deep Learning models**


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

# ─────────────────────────────────────────────────────────────────
# STEP 1: Prepare the dataset from store_merged
# ─────────────────────────────────────────────────────────────────

df = store_merged.copy()

# ── Fix corrupted DatetimeIndex ───────────────────────────────────
# The Date column was stored as integers (e.g. 20130101) or Unix nanoseconds
# and was misread by pandas. Reset and re-parse it correctly.

df = df.reset_index()   # Pull the broken index back into a column

# Identify the date column
date_col = None
for candidate in ['Date', 'date', 'index', 'level_0']:
    if candidate in df.columns:
        date_col = candidate
        break

print(f"Raw date sample (before fix): {df[date_col].iloc[:3].values}")

# Convert correctly: if values look like integers (20130101), use format
sample_val = str(df[date_col].iloc[0])
if 'NaT' not in sample_val and len(sample_val.replace('.','').replace('-','')) >= 8:
    try:
        # Try parsing as YYYYMMDD integer format first
        df[date_col] = pd.to_datetime(
            df[date_col].astype(str).str[:10].str.replace(r'\.\d+','', regex=True),
            infer_datetime_format=True
        )
    except Exception:
        df[date_col] = pd.to_datetime(df[date_col], unit='ns')

df = df.set_index(date_col)
df.index = pd.to_datetime(df.index)
df = df.sort_index()

print(f"Fixed date range : {df.index.min().date()} → {df.index.max().date()}")
print(f"Total rows       : {len(df):,}")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 2: Date-Boundary Split (60/20/20)
# ─────────────────────────────────────────────────────────────────

# ── Repair the index unconditionally ─────────────────────────────
# Reset whatever index exists and rebuild from the Date column
df = df.reset_index()   # Moves current index back to a column

# Identify the date column regardless of its current name
date_col = None
for candidate in ['Date', 'date', 'index', 'level_0']:
    if candidate in df.columns:
        try:
            df[candidate] = pd.to_datetime(df[candidate])
            date_col = candidate
            break
        except Exception:
            continue

assert date_col is not None, (
    "Cannot find a Date column. Available columns: " + str(df.columns.tolist())
)

df = df.set_index(date_col)          # Set Date as index
df.index = pd.to_datetime(df.index)  # Guarantee DatetimeIndex dtype
df = df.sort_index()                 # Enforce chronological order

print(f"Index type     : {type(df.index)}")
print(f"Full date range: {df.index.min().date()} → {df.index.max().date()}")
print(f"Total rows     : {len(df):,}")

# ── Date-boundary split (identical formula to SARIMA) ────────────
total_days = (df.index.max() - df.index.min()).days
print(f"Total calendar days : {total_days}")

assert total_days > 10, (
    f"total_days={total_days} is too small — the date index is likely wrong."
)

train_end = df.index.min() + pd.Timedelta(days=int(total_days * 0.60))
val_end   = df.index.min() + pd.Timedelta(days=int(total_days * 0.80))

train_df  = df[df.index <= train_end]
val_df    = df[(df.index > train_end) & (df.index <= val_end)]
test_df   = df[df.index > val_end]

print(f"\nBoundary dates:")
print(f"  train_end : {train_end.date()}")
print(f"  val_end   : {val_end.date()}")
print(f"\nSplit sizes:")
print(f"  Train : {len(train_df):,} rows | {train_df.index.min().date()} → {train_df.index.max().date()}")
print(f"  Val   : {len(val_df):,} rows   | {val_df.index.min().date()   if len(val_df)   > 0 else 'EMPTY ← PROBLEM'}")
print(f"  Test  : {len(test_df):,} rows  | {test_df.index.min().date()  if len(test_df)  > 0 else 'EMPTY ← PROBLEM'}")

assert len(train_df) > 1, "Train split is empty — check the Date column."
assert len(val_df)   > 1, f"Val split is empty — train_end={train_end.date()}, val_end={val_end.date()}"
assert len(test_df)  > 1, f"Test split is empty — val_end={val_end.date()}, df.max={df.index.max().date()}"

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 3: Sliding-Window Dataset Class
# ─────────────────────────────────────────────────────────────────

import numpy as np
from sklearn.preprocessing import MinMaxScaler

# ── 3.0  Build Sales_scaled and Store_idx on train_df/val_df/test_df
#         if they were not already created (these columns live on dl_df,
#         not on the general store_merged splits).
if 'Sales_scaled' not in train_df.columns:

    # -- log1p-transform Sales, then fit MinMaxScaler on train only
    _scaler_y = MinMaxScaler()

    _log_train = np.log1p(train_df['Sales'].values.astype('float32')).reshape(-1, 1)
    _log_val   = np.log1p(val_df['Sales'].values.astype('float32')).reshape(-1, 1)
    _log_test  = np.log1p(test_df['Sales'].values.astype('float32')).reshape(-1, 1)

    _scaler_y.fit(_log_train)   # fit on train only — no leakage

    train_df = train_df.copy()
    val_df   = val_df.copy()
    test_df  = test_df.copy()

    train_df['Sales_scaled'] = _scaler_y.transform(_log_train).ravel()
    val_df['Sales_scaled']   = _scaler_y.transform(_log_val).ravel()
    test_df['Sales_scaled']  = _scaler_y.transform(_log_test).ravel()

    print("Sales_scaled created via log1p + MinMaxScaler.")

else:
    print("Sales_scaled already present — skipping creation.")


if 'Store_idx' not in train_df.columns:

    # -- Re-index Store to 0-based integers required by nn.Embedding
    _all_stores  = sorted(train_df['Store'].unique())
    _store_map   = {s: i for i, s in enumerate(_all_stores)}

    train_df['Store_idx'] = train_df['Store'].map(_store_map).astype('int64')
    val_df['Store_idx']   = val_df['Store'].map(_store_map).fillna(0).astype('int64')
    test_df['Store_idx']  = test_df['Store'].map(_store_map).fillna(0).astype('int64')

    num_stores = len(_store_map)
    print(f"Store_idx created — {num_stores} unique stores.")

else:
    print("Store_idx already present — skipping creation.")


# Derive feature_cols from what is available in these splits
# (mirrors X_feature_cols used in dl_df but built from the current DataFrames)
if 'feature_cols' not in dir() or feature_cols is None:
    _drop   = {'Store', 'Sales', 'Customers', 'SalesPerCustomer',
               'Sales_scaled', 'Store_idx', 'Sales_log'}
    feature_cols = [c for c in train_df.columns if c not in _drop
                    and train_df[c].dtype in ('float32', 'float64',
                                              'int32', 'int64',
                                              'uint8', 'bool')]
    feature_cols = [c for c in feature_cols]
    print(f"feature_cols derived automatically: {len(feature_cols)} features")


# ── 3.1  Automatically select a safe SEQ_LEN that fits all three splits
_max_safe = min(len(train_df), len(val_df), len(test_df)) - 1
SEQ_LEN   = min(60, _max_safe)
print(f"\nVal rows   : {len(val_df):,}  |  Test rows : {len(test_df):,}")
print(f"Auto-selected SEQ_LEN = {SEQ_LEN}")

assert SEQ_LEN > 0, (
    "One of your splits has 0 or 1 rows — check your date boundaries in Step 2."
)


# ── 3.2  Dataset class
class RossmannDataset(Dataset):
    """
    Generates sliding-window sequences from the time-series DataFrame.
    Each sample:
        store_ids : (SEQ_LEN,)      integer store indices for embedding
        X         : (SEQ_LEN, F)    feature sequences
        y         : scalar float     scaled log-sales on the next step
    """
    def __init__(self, dataframe, feature_cols,
                 target_col='Sales_scaled', store_col='Store_idx', seq_len=SEQ_LEN):
        self.seq_len = seq_len
        self.X_arr   = dataframe[feature_cols].values.astype(np.float32)
        self.y_arr   = dataframe[target_col].values.astype(np.float32)
        self.sid_arr = dataframe[store_col].values.astype(np.int64)

    def __len__(self):
        return max(0, len(self.X_arr) - self.seq_len)

    def __getitem__(self, idx):
        X_seq   = self.X_arr[idx : idx + self.seq_len]
        sid_seq = self.sid_arr[idx : idx + self.seq_len]
        y_val   = self.y_arr[idx + self.seq_len]
        return (
            torch.tensor(sid_seq, dtype=torch.long),
            torch.tensor(X_seq,   dtype=torch.float32),
            torch.tensor(y_val,   dtype=torch.float32),
        )


# ── 3.3  Instantiate datasets
train_dataset = RossmannDataset(train_df, feature_cols)
val_dataset   = RossmannDataset(val_df,   feature_cols)
test_dataset  = RossmannDataset(test_df,  feature_cols)

print(f"\nTrain samples : {len(train_dataset):,}")
print(f"Val   samples : {len(val_dataset):,}")
print(f"Test  samples : {len(test_dataset):,}")


# ── 3.4  Auto-select BATCH_SIZE
BATCH_SIZE = min(512, len(train_dataset), len(val_dataset), len(test_dataset))
print(f"Auto-selected BATCH_SIZE = {BATCH_SIZE}")

assert BATCH_SIZE > 0, (
    "A dataset has 0 samples — SEQ_LEN is still too large for one of the splits."
)


# ── 3.5  DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True, drop_last=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True, drop_last=False)

print(f"\nTrain batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")

In [ ]:
import torch
import torch.nn as nn
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Training device:", device)

scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
class RossmannLSTM(nn.Module):
    def __init__(self, num_stores=1115, store_emb_dim=10,
                 num_cont_features=37,
                 hidden_size=128, num_layers=2):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Embedding for Store ID
        self.store_embedding = nn.Embedding(num_stores, store_emb_dim)

        # Standard LSTM layer: input size = store embedding dim + number of features
        self.lstm = nn.LSTM(
            input_size=store_emb_dim + num_cont_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        # Output layer: hidden_size only (not hidden_size * 2)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, store_ids, x_cont):
        # Embed store IDs → (batch, seq_len, store_emb_dim)
        store_emb = self.store_embedding(store_ids)

        # Concatenate embedding with input features
        x = torch.cat([store_emb, x_cont], dim=-1)

        # Initialize hidden and cell states
        # Shape: (num_layers, batch_size, hidden_size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        # LSTM forward pass; take the last timestep output
        out, _ = self.lstm(x, (h0, c0))
        return self.fc(out[:, -1, :])   # (batch, hidden_size) → (batch, 1)

# Instantiate the model
lstm_model = RossmannLSTM(
    num_stores=1115,
    num_cont_features=37
).to(device)

print(lstm_model)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 4: Instantiate the LSTM Model
# ─────────────────────────────────────────────────────────────────

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training device: {device}")

num_features = len(feature_cols)

model = RossmannLSTM(
    num_stores=num_stores,
    store_emb_dim=10,
    num_cont_features=num_features,
    hidden_size=128,
    num_layers=2
).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-5)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5
)

scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

print(model)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 5: Training Loop
# ─────────────────────────────────────────────────────────────────

num_epochs    = 40
best_val_loss = float('inf')
patience      = 5       # Early stopping patience
patience_ctr  = 0
train_losses  = []
val_losses    = []

for epoch in range(1, num_epochs + 1):

    # ── Training phase ────────────────────────────────────────────
    model.train()
    running_loss = 0.0

    for store_ids, seqs, labels in train_loader:
        # Move tensors to GPU (T4)
        store_ids = store_ids.to(device)
        seqs      = seqs.to(device)      # (batch, SEQ_LEN, 37)
        labels    = labels.to(device)    # (batch,)

        optimizer.zero_grad()

        # Forward pass under automatic mixed precision (float16 on GPU)
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            outputs = model(store_ids, seqs).squeeze(-1)  # (batch,)
            loss    = criterion(outputs, labels)

        # Backward pass with gradient scaling
        scaler.scale(loss).backward()
        # Gradient clipping — prevents exploding gradients in RNNs
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # ── Validation phase ──────────────────────────────────────────
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for store_ids, seqs, labels in val_loader:
            store_ids = store_ids.to(device)
            seqs      = seqs.to(device)
            labels    = labels.to(device)

            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                outputs   = model(store_ids, seqs).squeeze(-1)
                val_loss += criterion(outputs, labels).item()

    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    # Update learning-rate scheduler based on validation loss
    scheduler.step(avg_val_loss)

    print(f"Epoch [{epoch:3d}/{num_epochs}] | "
          f"Train Loss: {avg_train_loss:.6f} | "
          f"Val Loss: {avg_val_loss:.6f}")

    # ── Early stopping — save best weights ────────────────────────
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_ctr  = 0
        # Save the best model weights to restore later
        best_weights = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch}.")
            print(f"Restoring best weights (Val Loss: {best_val_loss:.6f})")
            model.load_state_dict(best_weights)
            break

print("\nTraining complete.")
print(f"Best Validation Loss: {best_val_loss:.6f}")

In [ ]:
!pip install torchmetrics

In [ ]:
import torchmetrics
import numpy as np
import torch

# ─────────────────────────────────────────────────────────────────
# EVALUATION: LSTM Model on Test Data — RMSE
# ─────────────────────────────────────────────────────────────────

rmse_metric = torchmetrics.MeanSquaredError(squared=False).to(device)

model.eval()
all_preds_lstm  = []
all_labels_lstm = []

with torch.no_grad():
    for store_ids, seqs, labels in test_loader:
        store_ids = store_ids.to(device)
        seqs      = seqs.to(device)
        labels    = labels.to(device)

        outputs = model(store_ids, seqs).squeeze(-1)

        rmse_metric(outputs, labels)

        all_preds_lstm.append(outputs.cpu())
        all_labels_lstm.append(labels.cpu())

rmse_scaled_lstm = rmse_metric.compute().item()
print(f"Test RMSE (scaled log space) : {rmse_scaled_lstm:.6f}")

all_preds_lstm  = torch.cat(all_preds_lstm).numpy().reshape(-1, 1)
all_labels_lstm = torch.cat(all_labels_lstm).numpy().reshape(-1, 1)

preds_log_lstm  = _scaler_y.inverse_transform(all_preds_lstm)
labels_log_lstm = _scaler_y.inverse_transform(all_labels_lstm)

preds_sales_lstm  = np.expm1(preds_log_lstm)
labels_sales_lstm = np.expm1(labels_log_lstm)

rmse_sales_lstm = np.sqrt(np.mean((preds_sales_lstm - labels_sales_lstm) ** 2))
print(f"Test RMSE (original Sales €) : {rmse_sales_lstm:,.2f}")

In [ ]:
lstm_train_losses = train_losses.copy()
lstm_val_losses = val_losses.copy()
rmse_scaled_lstm = rmse_scaled
rmse_sales_lstm  = rmse_sales

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Instantiate the GRU Model
# ─────────────────────────────────────────────────────────────────

import torch
import torch.nn as nn
import torch.optim as optim

class RossmannGRU(nn.Module):
    def __init__(self,
                 num_stores,
                 store_emb_dim,
                 num_cont_features,
                 hidden_size=128,
                 num_layers=2,
                 dropout=0.2):
        super().__init__()

        self.store_embedding = nn.Embedding(num_stores, store_emb_dim)

        self.gru = nn.GRU(
            input_size=store_emb_dim + num_cont_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, store_ids, x_cont):
        store_emb = self.store_embedding(store_ids)          # (B, T, emb_dim)
        x = torch.cat([store_emb, x_cont], dim=-1)           # (B, T, emb_dim + features)

        out, _ = self.gru(x)                                 # (B, T, hidden_size)
        last_hidden = out[:, -1, :]                          # (B, hidden_size)

        pred = self.fc(last_hidden)                          # (B, 1)
        return pred


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training device: {device}")

num_features = len(feature_cols)

model = RossmannGRU(
    num_stores=num_stores,
    store_emb_dim=10,
    num_cont_features=num_features,
    hidden_size=128,
    num_layers=2,
    dropout=0.2
).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5
)

scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

print(model)

In [ ]:
# ───────────────────────────────────────────────
# STEP 5: Training Loop (clean + robust)
# ───────────────────────────────────────────────
num_epochs    = 40
best_val_loss = float('inf')
patience      = 8          # a bit more patient
patience_ctr  = 0

train_losses = []
val_losses   = []

for epoch in range(1, num_epochs + 1):

    # ---- Training phase ----
    model.train()
    running_loss = 0.0

    for store_ids, seqs, labels in train_loader:
        store_ids = store_ids.to(device)
        seqs      = seqs.to(device)
        labels    = labels.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            outputs = model(store_ids, seqs).squeeze(-1)
            loss    = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    avg_train_loss = running_loss / max(1, len(train_loader))
    train_losses.append(avg_train_loss)

    # ---- Validation phase ----
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for store_ids, seqs, labels in val_loader:
            store_ids = store_ids.to(device)
            seqs      = seqs.to(device)
            labels    = labels.to(device)

            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                outputs = model(store_ids, seqs).squeeze(-1)
                val_loss += criterion(outputs, labels).item()

    avg_val_loss = val_loss / max(1, len(val_loader))
    val_losses.append(avg_val_loss)

    scheduler.step(avg_val_loss)

    print(
        f"Epoch [{epoch:2d}/{num_epochs}] | "
        f"Train Loss: {avg_train_loss:.6f} | "
        f"Val Loss: {avg_val_loss:.6f}"
    )

    # ---- Early stopping ----
    if avg_val_loss < best_val_loss - 1e-5:   # require a tiny real improvement
        best_val_loss = avg_val_loss
        patience_ctr   = 0
        best_weights   = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr >= patience:
            print(f"\nEarly stopping at epoch {epoch}.")
            print(f"Restoring best weights (Val Loss: {best_val_loss:.6f})")
            model.load_state_dict(best_weights)
            break

print("\nTraining complete.")
print(f"Best Validation Loss: {best_val_loss:.6f}")

In [ ]:
import torchmetrics
import numpy as np
import torch

# ─────────────────────────────────────────────────────────────────
# EVALUATION: GRU Model on Test Data — RMSE
# ─────────────────────────────────────────────────────────────────

rmse_metric = torchmetrics.MeanSquaredError(squared=False).to(device)

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for store_ids, seqs, labels in test_loader:
        store_ids = store_ids.to(device)
        seqs      = seqs.to(device)
        labels    = labels.to(device)

        outputs = model(store_ids, seqs).squeeze(-1)

        rmse_metric(outputs, labels)

        all_preds.append(outputs.cpu())
        all_labels.append(labels.cpu())

rmse_scaled = rmse_metric.compute().item()
print(f"Test RMSE (scaled log space) : {rmse_scaled:.6f}")

all_preds  = torch.cat(all_preds).numpy().reshape(-1, 1)
all_labels = torch.cat(all_labels).numpy().reshape(-1, 1)

preds_log  = _scaler_y.inverse_transform(all_preds)
labels_log = _scaler_y.inverse_transform(all_labels)

preds_sales  = np.expm1(preds_log)
labels_sales = np.expm1(labels_log)

rmse_sales = np.sqrt(np.mean((preds_sales - labels_sales) ** 2))
print(f"Test RMSE (original Sales €) : {rmse_sales:,.2f}")

In [ ]:
gru_train_losses = train_losses.copy()
gru_val_losses   = val_losses.copy()
rmse_scaled_gru  = rmse_scaled
rmse_sales_gru   = rmse_sales

## Step 8: Training and Validation Loss Curves

The loss curves below visualise the convergence behaviour of each model across epochs. A well-converged model exhibits decreasing training loss alongside a parallel validation loss, with no significant divergence between the two (which would indicate overfitting).

In [ ]:
def plot_loss_curves(train_losses, val_losses, model_name, ax):
    """Plot training and validation loss curves on a given matplotlib Axes object."""
    epochs = range(1, len(train_losses) + 1)

    ax.plot(
        epochs,
        train_losses,
        label='Training Loss',
        color='steelblue',
        linewidth=2
    )
    ax.plot(
        epochs,
        val_losses,
        label='Validation Loss',
        color='darkorange',
        linewidth=2,
        linestyle='--'
    )

    ax.set_title(f'{model_name} — Loss Curves', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('MSE Loss', fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)


# GRU vs LSTM
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_loss_curves(gru_train_losses, gru_val_losses, 'GRU', axes[0])
plot_loss_curves(lstm_train_losses, lstm_val_losses, 'LSTM', axes[1])

plt.suptitle(
    'Training and Validation Loss — GRU vs LSTM',
    fontsize=14,
    fontweight='bold',
    y=1.02
)

plt.tight_layout()
plt.savefig('loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Loss curves saved to loss_curves.png")

## Step 9: Evaluation on the Test Set

After training, predictions are generated on the held-out test set (final 20% by time). The inverse transformations are applied in reverse order — first `MinMaxScaler.inverse_transform`, then `numpy.expm1` — to recover predictions in original sales units (euros). Closed-day rows (`Open = 0`) are excluded from metric computation to prevent division-by-zero in percentage-based metrics.

The following plots compare predicted sales against actual sales on the test set. The first plot overlays predicted and actual values over time (first 500 samples for clarity). The second plot shows the scatter of predicted vs. actual values; a perfect model would produce all points on the 45-degree diagonal.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

def plot_predictions(y_true, y_pred, model_name, n_samples=500):
    """
    Generate two diagnostic plots:
    (1) Time-series overlay of actual vs predicted sales
    (2) Scatter plot of actual vs predicted sales
    """
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    n_samples = min(n_samples, len(y_true), len(y_pred))

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # ── Plot 1: Temporal overlay ──────────────────────────────────────────────
    axes[0].plot(
        y_true[:n_samples],
        label='Actual Sales',
        color='steelblue',
        linewidth=1.2,
        alpha=0.8
    )
    axes[0].plot(
        y_pred[:n_samples],
        label='Predicted Sales',
        color='darkorange',
        linewidth=1.2,
        alpha=0.8,
        linestyle='--'
    )
    axes[0].set_title(
        f'{model_name} — Actual vs Predicted (first {n_samples} samples)',
        fontsize=12,
        fontweight='bold'
    )
    axes[0].set_xlabel('Sample Index', fontsize=11)
    axes[0].set_ylabel('Sales (€)', fontsize=11)
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)
    axes[0].yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'{x:,.0f}')
    )

    # ── Plot 2: Scatter — Actual vs Predicted ─────────────────────────────────
    lim = max(y_true[:n_samples].max(), y_pred[:n_samples].max())

    axes[1].scatter(
        y_true[:n_samples],
        y_pred[:n_samples],
        alpha=0.3,
        s=8,
        color='steelblue'
    )
    axes[1].plot(
        [0, lim],
        [0, lim],
        'r--',
        linewidth=1.5,
        label='Perfect Prediction'
    )
    axes[1].set_title(
        f'{model_name} — Scatter: Actual vs Predicted',
        fontsize=12,
        fontweight='bold'
    )
    axes[1].set_xlabel('Actual Sales (€)', fontsize=11)
    axes[1].set_ylabel('Predicted Sales (€)', fontsize=11)
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)
    axes[1].xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'{x:,.0f}')
    )
    axes[1].yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'{x:,.0f}')
    )

    plt.tight_layout()
    file_name = f"{model_name.lower()}_predictions.png"
    plt.savefig(file_name, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Prediction plot saved to {file_name}")

## Residual Analysis

Residual analysis examines the distribution and pattern of prediction errors. Residuals should ideally be normally distributed around zero with no systematic pattern, confirming that the model has captured the underlying data structure without bias.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

def plot_residuals(y_true, y_pred, model_name):
    """Plot residual distribution histogram and residuals-over-index scatter."""
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    n = min(len(y_true), len(y_pred))
    residuals = y_true[:n] - y_pred[:n]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Histogram of residuals ────────────────────────────────────────────────
    axes[0].hist(
        residuals,
        bins=80,
        color='steelblue',
        edgecolor='white',
        alpha=0.85
    )
    axes[0].axvline(
        0,
        color='red',
        linestyle='--',
        linewidth=1.5,
        label='Zero residual'
    )
    axes[0].set_title(
        f'{model_name} — Residual Distribution',
        fontsize=12,
        fontweight='bold'
    )
    axes[0].set_xlabel('Residual (Actual − Predicted)', fontsize=11)
    axes[0].set_ylabel('Frequency', fontsize=11)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # ── Residuals over index ─────────────────────────────────────────────────
    axes[1].scatter(
        range(len(residuals)),
        residuals,
        alpha=0.2,
        s=5,
        color='steelblue'
    )
    axes[1].axhline(
        0,
        color='red',
        linestyle='--',
        linewidth=1.5
    )
    axes[1].set_title(
        f'{model_name} — Residuals over Test Samples',
        fontsize=12,
        fontweight='bold'
    )
    axes[1].set_xlabel('Sample Index', fontsize=11)
    axes[1].set_ylabel('Residual (€)', fontsize=11)
    axes[1].grid(True, alpha=0.3)
    axes[1].yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'{x:,.0f}')
    )

    plt.tight_layout()
    file_name = f'{model_name.lower()}_residuals.png'
    plt.savefig(file_name, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Residual plot saved to {file_name}")

In [ ]:
# labels_sales, preds_sales computed after inverse transform
plot_residuals(labels_sales, preds_sales, 'GRU')


In [ ]:
plot_residuals(labels_sales_lstm, preds_sales_lstm, 'LSTM')

# **Model Comparison — SARIMA vs GRU vs LSTM**

In [ ]:
# ── Compile comparison table including SARIMA RMSE baseline ─────────────────
# Note: Replace the SARIMA RMSE value below with the actual result from your model above.
comparison_data = {
    'Model'    : ['SARIMA (Baseline)', 'GRU', 'LSTM'],
    'RMSE'     : [rms_test_scaled,       rmse_scaled,       rmse_scaled_lstm],
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df[['RMSE']] = \
    comparison_df[['RMSE']].apply(
        pd.to_numeric, errors='coerce'
    ).round(4)

print("\n" + "="*65)
print("    Model Performance Comparison on Test Set")
print("="*65)
print(comparison_df.to_string(index=False))
print("="*65)
print("Lower values indicate superior forecasting performance.")


## Comparison Bar Chart

In [ ]:
# ── Bar chart comparing RMSE across all three models ─────────────────────────
models = ['SARIMA', 'GRU', 'LSTM']
rmse_values = [rms_test_scaled, rmse_scaled, rmse_scaled_lstm]
colors = ['#b0bec5', '#42a5f5', '#66bb6a']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(models, rmse_values, color=colors, edgecolor='white', width=0.5)

# Small dynamic offset based on chart scale
offset = max(rmse_values) * 0.02

# Annotate each bar with its RMSE value
for bar, val in zip(bars, rmse_values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + offset,
        f'{val:.2f}',
        ha='center',
        va='bottom',
        fontsize=11,
        fontweight='bold'
    )

ax.set_title('RMSE Comparison: SARIMA vs GRU vs LSTM', fontsize=13, fontweight='bold')
ax.set_ylabel('RMSE', fontsize=11)
ax.set_xlabel('Model', fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, max(rmse_values) * 1.12)

plt.tight_layout()
plt.savefig('rmse_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("RMSE comparison chart saved to rmse_comparison.png")